In [2]:
from pathlib import Path
import cv2
import numpy as np

# --------------------------------------------------
# YOLO utilities
# --------------------------------------------------
def read_yolo_txt(txt_path):
    """
    Reads YOLO rows:
    cls xc yc w h [conf]
    """
    rows = []
    with open(txt_path, "r") as f:
        for line in f:
            p = line.strip().split()
            if len(p) < 5:
                continue
            rows.append({
                "cls": int(float(p[0])),
                "xc": float(p[1]),
                "yc": float(p[2]),
                "w":  float(p[3]),
                "h":  float(p[4]),
                "conf": float(p[5]) if len(p) > 5 else None
            })
    return rows

def yolo_to_xyxy(xc, yc, w, h, W, H):
    bw, bh = w * W, h * H
    x1 = xc * W - bw / 2
    y1 = yc * H - bh / 2
    x2 = xc * W + bw / 2
    y2 = yc * H + bh / 2
    return x1, y1, x2, y2

def pad_xyxy(x1, y1, x2, y2, W, H, pad_ratio=0.4, min_pad=10):
    bw, bh = (x2 - x1), (y2 - y1)
    px = max(bw * pad_ratio, min_pad)
    py = max(bh * pad_ratio, min_pad)
    return (
        max(0, int(x1 - px)),
        max(0, int(y1 - py)),
        min(W, int(x2 + px)),
        min(H, int(y2 + py))
    )

def crop(img, box):
    x1, y1, x2, y2 = box
    return img[y1:y2, x1:x2]

def stitch(pre_crop, post_crop, target_h=1024, pad_value=0, interp_up=cv2.INTER_CUBIC):
    """Stitch PRE/POST crops side-by-side while keeping crops sharp.

    Key idea:
      1) NEVER resize the crops to match each other (avoids blur).
      2) Pad to equal height.
      3) Optionally upscale the final stitched pair to a target height (e.g., 1024).

    Args:
        pre_crop, post_crop: HxWxC uint8 crops.
        target_h: final output height. If None, no upscaling is applied.
        pad_value: background padding value (0 = black).
        interp_up: interpolation used ONLY for the final upscale.
    """
    # 1) Pad crops to the same height (no interpolation)
    H = max(pre_crop.shape[0], post_crop.shape[0])

    def pad_to_h(im, H):
        h, w = im.shape[:2]
        if h == H:
            return im
        pad_total = H - h
        pad_top = pad_total // 2
        pad_bot = pad_total - pad_top
        return cv2.copyMakeBorder(
            im, pad_top, pad_bot, 0, 0,
            borderType=cv2.BORDER_CONSTANT,
            value=(pad_value, pad_value, pad_value)
        )

    pre_p  = pad_to_h(pre_crop, H)
    post_p = pad_to_h(post_crop, H)

    # 2) Concatenate
    canvas = np.concatenate([pre_p, post_p], axis=1)

    # 3) Add labels (safe even after upscaling)
    #cv2.putText(canvas, "PRE",  (10, 30), cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)
    #cv2.putText(canvas, "POST", (pre_p.shape[1] + 10, 30),
    #           cv2.FONT_HERSHEY_SIMPLEX, 1, (255,255,255), 2)

    # 4) Optional upscale to a consistent VLM-friendly size
    if target_h is not None and canvas.shape[0] != target_h:
        scale = target_h / canvas.shape[0]
        new_w = max(1, int(round(canvas.shape[1] * scale)))
        canvas = cv2.resize(canvas, (new_w, target_h), interpolation=interp_up)

    return canvas

# --------------------------------------------------
# Main pipeline
# --------------------------------------------------
def build_vlm_pairs(
    pre_img_path,
    post_img_path,
    yolo_txt,
    out_dir="crop_images/pair_00000048", #change to output directory path
    pad_ratio=0.3,
    conf_thresh=0.2,
    target_h=1024
):
    out_dir = Path(out_dir)
    out_dir.mkdir(parents=True, exist_ok=True)

    pre  = cv2.imread(str(pre_img_path))
    post = cv2.imread(str(post_img_path))
    assert pre is not None and post is not None

    H_pre, W_pre   = pre.shape[:2]
    H_post, W_post = post.shape[:2]

    rows = read_yolo_txt(yolo_txt)

    k = 0
    for r in rows:
        if r["conf"] is not None and r["conf"] < conf_thresh:
            continue

        # PRE box
        x1p, y1p, x2p, y2p = yolo_to_xyxy(r["xc"], r["yc"], r["w"], r["h"], W_pre, H_pre)
        pre_box = pad_xyxy(x1p, y1p, x2p, y2p, W_pre, H_pre, pad_ratio)

        # POST box (same normalized coords, different size)
        x1q, y1q, x2q, y2q = yolo_to_xyxy(r["xc"], r["yc"], r["w"], r["h"], W_post, H_post)
        post_box = pad_xyxy(x1q, y1q, x2q, y2q, W_post, H_post, pad_ratio)

        pre_crop  = crop(pre, pre_box)
        post_crop = crop(post, post_box)

        pair = stitch(pre_crop, post_crop, target_h=target_h)

        tag = f"b{k:03d}_conf{r['conf']:.2f}"
        cv2.imwrite(str(out_dir / f"{tag}_pre.png"), pre_crop)
        cv2.imwrite(str(out_dir / f"{tag}_post.png"), post_crop)
        cv2.imwrite(str(out_dir / f"{tag}_pair.png"), pair)
        k += 1

    print(f"Saved {k} VLM-ready building pairs to {out_dir.resolve()}")


In [ ]:
build_vlm_pairs(
    pre_img_path = './sample_images/moore-tornado_00000048_pre_disaster.png', #change to pre-disaster image path
    post_img_path = './sample_images/moore-tornado_00000048_post_disaster.png', #change to post-disaster image path
    yolo_txt = './sample_images/labels/moore-tornado_00000048_pre_disaster.txt' ##change to YOLO prediction annotation labels path
)